# BERT Fine-tuning Template

General-purpose fine-tuning scaffold for sequence classification / regression with BERT.

**Sections**
1. Imports & Config
2. 📥 Data Input *(placeholder)*
3. 🔧 Data Preprocessing *(placeholder)*
4. Dataset & DataLoader
5. Model
6. Optimizer & Scheduler
7. Training Loop
8. Evaluation
9. Inference

## 0. Install dependencies

In [ ]:
!pip install transformers datasets torch scikit-learn tqdm -q

## 1. Imports & Config

In [ ]:
import torch
import torch.nn as nn
import numpy as np
from torch.utils.data import Dataset, DataLoader
from transformers import (
    BertTokenizer,
    BertForSequenceClassification,
    get_linear_schedule_with_warmup,
)
from sklearn.metrics import classification_report
from tqdm.auto import tqdm

# ── Config ────────────────────────────────────────────────────────────────────
MODEL_NAME   = "bert-base-uncased"   # backbone; swap as needed
MAX_LEN      = 128                   # max token length
BATCH_SIZE   = 16
EPOCHS       = 3
LR           = 2e-5
WARMUP_RATIO = 0.1
NUM_LABELS   = 2                     # number of output classes / dims
TASK         = "classification"      # "classification" | "regression"
DEVICE       = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print(f"Device : {DEVICE}")
print(f"Task   : {TASK}  |  Labels: {NUM_LABELS}")

---
## 2. 📥 Data Input

> **PLACEHOLDER** — Load your raw data here.
> Expected outputs of this section:
> -  : list / DataFrame of training samples
> -    : list / DataFrame of validation samples
> -   : list / DataFrame of test samples (optional)
>
> Each sample must eventually provide a **text** field and a **label** field.

In [ ]:
# ── PLACEHOLDER: replace the block below with your data source ─────────────────
#
# Option A — local CSV / TSV
# import pandas as pd
# df = pd.read_csv("path/to/data.csv")
# raw_train = df[df["split"] == "train"]
# raw_val   = df[df["split"] == "val"]
# raw_test  = df[df["split"] == "test"]
#
# Option B — HuggingFace datasets
# from datasets import load_dataset
# ds = load_dataset("<dataset_name>")
# raw_train, raw_val, raw_test = ds["train"], ds["validation"], ds["test"]
#
# Option C — JSON / JSONL
# import json
# with open("path/to/data.jsonl") as f:
#     records = [json.loads(l) for l in f]
#
# ────────────────────────────────────────────────────────────────────────────

raw_train = None   # TODO
raw_val   = None   # TODO
raw_test  = None   # TODO  (set to None if no test split)

# Quick sanity check — uncomment after filling in above
# print(f"Train: {len(raw_train)}  Val: {len(raw_val)}")
# print("Sample:", raw_train[0])

---
## 3. 🔧 Data Preprocessing

> **PLACEHOLDER** — Transform raw data into parallel lists of texts and labels.
> Expected outputs of this section:
> - , 
> - ,   
> - ,   (optional)
>
>   → 
>  →  for classification,  for regression

In [ ]:
# ── PLACEHOLDER: implement cleaning / label mapping here ───────────────────
#
# LABEL_MAP = {"negative": 0, "positive": 1}   # adjust to your label space
#
# def clean_text(text: str) -> str:
#     text = text.strip()
#     # add further cleaning steps (lower-case, remove HTML, etc.)
#     return text
#
# def extract(split_data):
#     texts  = [clean_text(row["text"])  for row in split_data]
#     labels = [LABEL_MAP[row["label"]]  for row in split_data]
#     return texts, labels
#
# train_texts, train_labels = extract(raw_train)
# val_texts,   val_labels   = extract(raw_val)
# test_texts,  test_labels  = extract(raw_test) if raw_test else ([], [])
#
# ────────────────────────────────────────────────────────────────────────────

train_texts, train_labels = [], []   # TODO
val_texts,   val_labels   = [], []   # TODO
test_texts,  test_labels  = [], []   # TODO (leave empty if no test split)

# Quick sanity check — uncomment after filling in above
# print(f"Train: {len(train_texts)}  Val: {len(val_texts)}")
# print("Sample text  :", train_texts[0])
# print("Sample label :", train_labels[0])

---
## 4. Dataset & DataLoader

In [ ]:
tokenizer = BertTokenizer.from_pretrained(MODEL_NAME)


class TextDataset(Dataset):
    def __init__(self, texts, labels):
        self.texts  = texts
        self.labels = labels

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        enc = tokenizer(
            self.texts[idx],
            max_length=MAX_LEN,
            padding="max_length",
            truncation=True,
            return_tensors="pt",
        )
        label = self.labels[idx]
        label_tensor = (
            torch.tensor(label, dtype=torch.long)
            if TASK == "classification"
            else torch.tensor(label, dtype=torch.float)
        )
        return {
            "input_ids":      enc["input_ids"].squeeze(0),
            "attention_mask": enc["attention_mask"].squeeze(0),
            "token_type_ids": enc["token_type_ids"].squeeze(0),
            "labels":         label_tensor,
        }


train_ds = TextDataset(train_texts, train_labels)
val_ds   = TextDataset(val_texts,   val_labels)
test_ds  = TextDataset(test_texts,  test_labels) if test_texts else None

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,  num_workers=0)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
test_loader  = DataLoader(test_ds,  batch_size=BATCH_SIZE, shuffle=False, num_workers=0) if test_ds else None

print(f"Train batches: {len(train_loader)}  |  Val batches: {len(val_loader)}")

---
## 5. Model

In [ ]:
# BertForSequenceClassification handles the [CLS] → Linear head internally.
# For regression set num_labels=1; the model then uses MSELoss automatically.
model = BertForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=NUM_LABELS if TASK == "classification" else 1,
    hidden_dropout_prob=0.1,
    attention_probs_dropout_prob=0.1,
).to(DEVICE)

total_params     = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Total params    : {total_params:,}")
print(f"Trainable params: {trainable_params:,}")

---
## 6. Optimizer & Scheduler

In [ ]:
optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=0.01)

total_steps  = len(train_loader) * EPOCHS
warmup_steps = int(total_steps * WARMUP_RATIO)

scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=warmup_steps,
    num_training_steps=total_steps,
)

print(f"Total steps: {total_steps}  |  Warmup steps: {warmup_steps}")

---
## 7. Training Loop

In [ ]:
def train_epoch(model, loader, optimizer, scheduler):
    model.train()
    total_loss = 0.0
    pbar = tqdm(loader, desc="  train", leave=False)
    for batch in pbar:
        input_ids      = batch["input_ids"].to(DEVICE)
        attention_mask = batch["attention_mask"].to(DEVICE)
        token_type_ids = batch["token_type_ids"].to(DEVICE)
        labels         = batch["labels"].to(DEVICE)

        optimizer.zero_grad()
        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            token_type_ids=token_type_ids,
            labels=labels,
        )
        loss = outputs.loss
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        scheduler.step()

        total_loss += loss.item()
        pbar.set_postfix(loss=f"{loss.item():.4f}")

    return total_loss / len(loader)


@torch.no_grad()
def evaluate(model, loader):
    model.eval()
    total_loss = 0.0
    all_preds, all_labels = [], []

    for batch in tqdm(loader, desc="  eval", leave=False):
        input_ids      = batch["input_ids"].to(DEVICE)
        attention_mask = batch["attention_mask"].to(DEVICE)
        token_type_ids = batch["token_type_ids"].to(DEVICE)
        labels         = batch["labels"].to(DEVICE)

        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            token_type_ids=token_type_ids,
            labels=labels,
        )
        total_loss += outputs.loss.item()

        if TASK == "classification":
            preds = outputs.logits.argmax(dim=-1).cpu().numpy()
        else:
            preds = outputs.logits.squeeze(-1).cpu().numpy()

        all_preds.append(preds)
        all_labels.append(labels.cpu().numpy())

    all_preds  = np.concatenate(all_preds)
    all_labels = np.concatenate(all_labels)
    avg_loss   = total_loss / len(loader)

    if TASK == "classification":
        report = classification_report(all_labels, all_preds, zero_division=0, output_dict=True)
        return avg_loss, report["accuracy"], report["macro avg"]["f1-score"]
    else:
        from scipy.stats import pearsonr
        r, _ = pearsonr(all_preds, all_labels)
        mse  = float(np.mean((all_preds - all_labels) ** 2))
        return avg_loss, r, mse


# ── Main training loop ───────────────────────────────────────────────────────────────────
best_val_loss = float("inf")

for epoch in range(1, EPOCHS + 1):
    print(f"
Epoch {epoch}/{EPOCHS}")
    train_loss = train_epoch(model, train_loader, optimizer, scheduler)

    if TASK == "classification":
        val_loss, accuracy, macro_f1 = evaluate(model, val_loader)
        print(f"  Train loss : {train_loss:.4f}")
        print(f"  Val loss   : {val_loss:.4f}")
        print(f"  Accuracy   : {accuracy:.4f}")
        print(f"  Macro-F1   : {macro_f1:.4f}")
    else:
        val_loss, pearson_r, mse = evaluate(model, val_loader)
        print(f"  Train loss : {train_loss:.4f}")
        print(f"  Val loss   : {val_loss:.4f}")
        print(f"  Pearson r  : {pearson_r:.4f}")
        print(f"  MSE        : {mse:.4f}")

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        torch.save(model.state_dict(), "demo_best_model.pt")
        print("  ✓ Saved best model")

---
## 8. Evaluation on Test Set

In [ ]:
if test_loader is not None:
    model.load_state_dict(torch.load("demo_best_model.pt", map_location=DEVICE))

    if TASK == "classification":
        test_loss, accuracy, macro_f1 = evaluate(model, test_loader)
        print(f"Test loss  : {test_loss:.4f}")
        print(f"Accuracy   : {accuracy:.4f}")
        print(f"Macro-F1   : {macro_f1:.4f}")
    else:
        test_loss, pearson_r, mse = evaluate(model, test_loader)
        print(f"Test loss  : {test_loss:.4f}")
        print(f"Pearson r  : {pearson_r:.4f}")
        print(f"MSE        : {mse:.4f}")
else:
    print("No test split provided — skipping.")

---
## 9. Inference

In [ ]:
model.load_state_dict(torch.load("demo_best_model.pt", map_location=DEVICE))
model.eval()


@torch.no_grad()
def predict(text: str):
    enc = tokenizer(
        text,
        max_length=MAX_LEN,
        padding="max_length",
        truncation=True,
        return_tensors="pt",
    ).to(DEVICE)

    logits = model(**enc).logits

    if TASK == "classification":
        probs = torch.softmax(logits, dim=-1).squeeze(0).cpu().numpy()
        pred  = int(np.argmax(probs))
        print(f"Text  : {text!r}")
        print(f"Class : {pred}  (probs: {probs.round(3)})")
    else:
        score = logits.squeeze(-1).item()
        print(f"Text  : {text!r}")
        print(f"Score : {score:.4f}")


# ── Demo calls (replace with your own examples) ──────────────────────────────
predict("Replace this with a real example from your dataset.")
predict("Another example sentence for inference.")